In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from vncorenlp import VnCoreNLP
import pandas as pd
import re
import os

llm = ChatOllama(model="myaniu/qwen2.5-1m")  # hoặc "llama3:8b-instruct"

def summary_abstractive(text, grade, length):
    prompt = [
        SystemMessage(content=f"Bạn là giáo viên dạy môn học tiếng Việt cho học sinh tiểu học tại Việt Nam."),
        HumanMessage(content=f"""
        NHIỆM VỤ:
            Tóm tắt diễn giải đoạn văn sau cho học sinh lớp {grade}.
        YÊU CẦU BẮT BUỘC:
            - Viết ĐÚNG {length} câu, không nhiều hơn, không ít hơn
            - Giữ đúng, đầy đủ các ý chính, ý phụ, nội dung của văn bản gốc
            - Ngôn ngữ đơn giản, tự nhiên, phù hợp học sinh lớp {grade}.
            - Các câu từ phải đảm bảo tính chính xác, tính logic, liên kết và mạch lạc
            - Viết trên MỘT DÒNG
            - KHÔNG thêm tiêu đề, chú thích, ký hiệu, lời dẫn hay nhận xét
        ĐOẠN VĂN:
        {text}

        KẾT QUẢ:
        """)
    ]
    response = llm(prompt)
    return response.content.strip()

vncorenlp = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg"
)

def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)
    tokens = []
    for sent in sentences:
        tokens.extend(sent)
    return tokens

def load_grade_vocab(grade, vocab_dir="../../../datasets/grade_vocab"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab

def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0
    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)


data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_output_safe1_final copy.xlsx")
df = data.copy()
max_retry = 5
threshold = 0.85 #giai thich pp nao
annotator = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

for idx, row in df.iterrows():
    if pd.isna(row["summary"]) or row["summary"] == "":
        content = str(row["content"])
        grade = int(row["grade"])
        sentences = annotator.tokenize(content)
        num_sentences = len(sentences)
        length = num_sentences // 3
        vocab = load_grade_vocab(grade)
        dict_result = {}
        score = 0
        for i in range(max_retry):
            summary_result = summary_abstractive(content, grade, length)
            score = vocab_coverage(summary_result, vocab)
            if score >= threshold:
                df.loc[idx, "summary"] = summary_result
                break
            else:
                df.loc[idx, "summary"] = ""

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_output_safe1_final copy1.xlsx", index=False)

C:\Users\minhv\AppData\Local\Temp\ipykernel_14948\2960101606.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Cá chép và gà nhí cùng thi vẽ. Cá chép vẽ vua nước, còn gà nhí vẽ mẹ gà chăm sóc đàn con. Cô cò và chú trắm chấm, họ khen bức vẽ của gà nhí vừa đẹp vừa có ý nghĩa.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, "summary"] = summary_result


KeyboardInterrupt: 

In [3]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from vncorenlp import VnCoreNLP
from collections import defaultdict
import numpy as np
# Khởi tạo VnCoreNLP
annotator = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

def sentence_tokenize_vn(text):
    sentences = []
    for sent in annotator.tokenize(text):
        sentences.append(" ".join(sent))
    return sentences
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    "vinai/phobert-base",
    use_fast=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

encoder = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
encoder.eval()
@torch.no_grad()
def score_sentence(sentence):
    encoding = tokenizer(
        sentence,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    outputs = encoder(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    sent_emb = outputs.last_hidden_state.mean(dim=1)
    score = sent_emb.norm(p=2).item()   # L2-norm làm proxy score
    return score
def sliding_window(sentences, window_size=5, stride=2):
    windows = []
    for i in range(0, len(sentences), stride):
        window = sentences[i:i + window_size]
        if len(window) == window_size:
            windows.append((i, window))
    return windows
def extractive_summary(
    text,
    window_size=5,
    stride=2,
    extract_ratio=0.25
):
    sentences = sentence_tokenize_vn(text)

    if len(sentences) == 0:
        return ""

    windows = sliding_window(sentences, window_size, stride)

    sentence_scores = defaultdict(list)

    for start_idx, window in windows:
        for j, sentence in enumerate(window):
            score = score_sentence(sentence)
            sentence_scores[start_idx + j].append(score)

    # Voting / Aggregation
    final_scores = {
        idx: np.mean(scores)
        for idx, scores in sentence_scores.items()
    }

    # Chọn top-k
    k = max(1, int(len(final_scores) * extract_ratio))

    top_sentences = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:k]

    # Giữ thứ tự gốc
    top_sentences = sorted(top_sentences, key=lambda x: x[0])

    summary = " ".join([sentences[idx] for idx, _ in top_sentences])
    return summary


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [10]:
text = """
Khi khác được mùa sẽ chờ nộp cũng chưa muộn " ' . Sứ họ Trịnh đành phải quay về . Neu nghĩ đến công tổ _ tiên ta , nên cắt cả Nghệ _ An cho ta nữa , huống _ chi là đất Thuận _ Quảng " ’ . Lần này chúa Trịnh không chịu được nữa bèn phát quân . Quân hai bên đối luỹ nhau cùng cầm _ cự . Đêm đến , lợi _ dụng nước thuỷ _ triều lên , quân Nguyễn__ bắn vào dinh của Nguyễn__ Khải , quân Trịnh sợ , rối _ loạn . Cánh quân_chủ _ lực của Trịnh _ Tráng đang rất mạnh vừa kéo đến , quân Nguyễn__ đánh không lại . Quân Trịnh thừa _ thắng cướp của _ cải . Sau đó quân Nguyễn_đem tượng _ binh đánh chặn ngang làm quân Trịnh tan võ , quân _ lính chết rất nhiều . Nguyễn__ Hữu _ Dật mưu vói Truong _ Phước _ Da sai gián _ điệp phao _ đồn rằng anh _ em Trịnh _ Gia , Trịnh _ Nhạc mưu nổi _ loạn . Neu không nghe mệnh thì ta đem quân đánh là có danh _ nghĩa " . Trịnh _ Tráng liền sai Lại bộ Thượng _ thư Nguyễn__ Khác _ Minh đcm sắc tiến phong chúa Sãi làm Tiết _ chế Thuận _ Hoá - Quảng _ Nam nhị xứ thuỷ _ bộ chư dinh kiêm Tổng nội _ ngoại bình chương quân _ quốc ưọng sự Thái _ phó công và giục đến Đông _ Đô để đi đánh ưiều Mạc ở Cao _ Bằng . Nghe vậy , chúa Sãi quyết _ định tạm nhận sắc để họ Trịnh không ngờ và có thời _ gian phòng _ thủ chuẩn _ bị lực _ lượng đối _ phó rồi sẽ dùng kế trả lại sắc . Hai bên đánh nhau quyết _ liệt . Đình _ Hùng chém tướng Trịnh tại _ trận rồi chiếm _ giữ đất , lập làm dinh Bố _ Chính ( gọi là Dinh _ Ngói ) , lấy Trương _ Phước _ Phấn ( con Trương _ Phước _ Da ) ưấn giữ để chống _ trả quân Trịnh . Lại đặt xích sắt chắn ngang các cùa biển Nhật _ Lệ và Minh _ Linh ’ ( cửa Đồng Hói và cửa _ Tùng ) . Khi lực _ lượng đã được chuẩn _ bị , phòng _ tuyến tự _ vệ đã hoàn _ thiện , Nguyễn__ Phúc _ Nguyên đã cùng với Đào _ Duy _ Từ ngày _ đêm mưu _ tính chống họ Trịnh ' . Chúa _ Nguyễn_đã cho mộ thêm những người khoẻ _ mạnh làm _ thân _ binh . Chúa _ Nguyễn_còn cho mộ thêm người có sức _ mạnh và am _ hiểu võ _ nghệ ờ hai xứ Thuận - Quảng bổ làm _ thân _ binh ở các cơ , đội , người có công cũng được lục dụng ’ . Anh tức _ thì sai người đi quy _ thuận với họ Trịnh * . Nguyễn__ Phúc _ Nguyên sai Đại _ tướng Nguyễn__ Mỹ _ Thắng và Đốc _ chiến Nguyễn__ Hữu _ Dật đem quân chống lại . Nguyễn__ Hữu _ Tiến thì cho đắp luỹ Trường _ Sa để bảo _ vệ luỹ chính Quân Trịnh bắn súng làm hiệu nhưng không thấy Anh tiếp _ ứng . Hơn một tuần không thấy nội _ ứng , quân Trịnh chán _ nản , quân Nguyễn__ xông ra đánh . Quân Trịnh tháo _ chạy , chết quá nửa ' . Chúa _ Thượng lệnh cho Nguyễn__ Phúc _ Khê sai các tướng Bùi__ Hùng _ Lương , Tống _ Triều _ Phương lĩnh thuỷ _ binh ; Tôn _ Thất_Yên , Tống _ Văn _ Hùng lãnh bộ _ binh cùng tiến đánh , lấy được sổ đồng _ tâm , bắt được Anh và đồng _ bọn đem giết ’ . Tướng cùa họ Nguyễn_là Nguyễn__ Hữu _ Dật đã làm kế phản _ gián , đưa thư cho họ Trịnh nói Khắc _ Liệt đã cùng quân Nguyễn__ ước _ hẹn giá thua đé dụ Trịnh _ Tráng đem quân vào đánh , rồi sẽ hợp với quân Nguyễn__ đánh Tráng . Cả hai đều bị trúng kế . Quân Nguyễn__ liền chiếm châu Bắc _ Bố _ Chính . Sau đó , Trịnh _ Tráng có gùi thư nói về tình__ nghĩa lâu _ đời , quân Nguyễn__ lại trả châu Bắc _ Bố _ Chính cho quân Trịnh . Nhưng sau đó , chúa Nguyễn thấy tình _ hình thuận _ lợi , " nước _ nhà phong _ phú " ‘ , " vẫn có ý đánh miền Bắc " ’ , đã ráo _ riết cho kén _ chọn bộ _ binh , thao _ diễn trận pháp ’ . Tướng đứng đầu của quân Nguyễn_là Bùi__ Công _ Thắng cố đánh bị chết . Quân Trịnh đánh chiếm cùa Nhật _ Lệ . Quân Nguyễn__ phòng _ thủ rất vững . Quân Trịnh đánh không được . Tống _ Thị còn xin đem gia sàn vào giúp _ việc quân _ lương . Quân Trịnh tiến vào xâm _ chiếm dinh Quảng _ Bình . Ký _ lục là Thịnh _ Hội phải trốn chạy qua sông . Quân Trịnh đánh mãi không được . Quân Trịnh bị đánh bất _ ngờ , thua chạy , lại bị quân thuỷ đánh truy _ kích , đều bị chết _ đuối . Quân Nguyễn__ đại _ thắng , bắt sống được ba tướng của quân Trịnh là Gia , Lý , Mỹ và 3 vạn tàn _ quân ’ . Chúa sai Nguyễn__ Hữu _ Tiến làm Tiết _ chế và Nguyễn__ Hữu _ Dật làm Đốc _ chiến . Trấn _ thủ dinh Bố _ Chính là Phù _ Dương ra Phù _ Lưu phá dinh Tam _ Hiệu ( Ba _ Đồn ) . Phạm _ Tất _ Đồng thua chạy vào Lũng _ Bồng . Trịnh _ Đào nghe tin Tam _ Hiệu thất _ thủ đã đem quân vào cứu _ viện . Nguyễn__ Hữu _ Tiến tiến _ quân đến Thạch _ Hà , Tham đốc bên Trịnh là Đặng__ Minh _ Tắc phải ra hàng . Nguyễn__ Hữu _ Tiến và Nguyễn__ Hữu _ Dật sai các tướng chia thành các đạo cùng tiến . Chính _ đạo do Trương _ Phúc _ Hùng , Phù _ Dương , Thuần _ Đức , Khuê _ Thắng đốc _ suất quân tiên _ phong đánh chặn du binh của quân Trịnh , đến thẳng Lạc _ Xuyên _ Hạ , đánh _ phá dinh Trịnh _ Trượng . Nguyễn__ Văn _ Thiêm vẫn giữ chức Đốc _ suất thuỷ _ quân cùng Dương _ Hổ làm Đốc thị đem quân đóng ờ Kỳ _ Hoa . Hải _ Dương thì không nộp thuế cho tuyệt lương . Tướng Trịnh là Thân _ Văn _ Quanh thua chạy . Nguyễn__ Hữu _ Dật đốc quân đến gặp tướng Trịnh là Võ _ Văn _ Thiêm , Văn _ Thiêm lui về An _ Trường . Hữu _ Tiến xuất _ quân_chính _ đạo đến Minh _ Lưontig . Tống _ Hữu _ Đại xuất _ quân đến núi Bình _ Lãng . Đào _ Quang _ Nhiêu chia quân chống đánh . Quân Trịnh bị tướng Nguyễn_là Đăng _ Doanh đánh thua , các tướng Trịnh đều bỏ trốn . Quang _ Nhiêu cũng bỏ ữốn về An _ Trường . Hữu _ Tiến và Hữu _ Dật thu _ quân đóng ở Vân _ Cát . Khi nghe các tướng đã lui về Hà _ Trung , chúa bèn dừng lại xã An _ Trạch , Nguyễn__ Hữu _ Dật đến yết _ kiến . Chúa chuẩn _ y cho kén _ chọn tướng . Bộ _ hạ của Ninh là Trịnh _ Bàn và Trương _ Đắc _ Doanh đã quy _ hàng Nguyễn__ Hữu _ Tiến . Họ Nguyễn_vì đường _ sá vận _ tải lương _ thảo khó _ khăn đã cho lập tuyển trường ờ Nghệ _ An , duyệt lấy 3 hạng tráng , quân và dân , thu thuế _ thân để phòng cấp _ phát và sai quan thu tô ruộng thực canh tại 7 huyện mới chiếm ở Nghệ _ An để cấp lương cho quân ' . Nguyễn__ Hữu _ Dật biết mưu ấy cho tướng là Trương _ Văn _ Vân phục quân ở rừng Thạch _ Hiệp . Triệu _ Tô và Triệu _ Tú _ Minh đóng ờ gò cao Hoành _ Càng để chống _ cự . Quân Trịnh do Đô _ đốc Diệu dần quân đang đêm đem quân đến rừng Hoành _ Luỹ , bị phục _ binh của quân Nguyễn__ chống lại , quân Trịnh tan võ chết rất nhiều . Hai bên Trịnh và Nguyễn__ đóng quân đối _ diện ở hai bên bờ sông đe cầm _ cự với nhau . Hữu _ Tiến cho quân Nguyễn__ qua sông , đánh vào Mỹ _ Dụ , Trịnh _ Khiêm bị thua . Chúa _ Trịnh _ Căn họp các tướng bàn hỏi mưu _ kế . Chúa chia quân làm 2 đạo . Một đạo do Hoàng _ Nghĩa _ Giao và Phan _ Kiệm _ Toàn dẫn _ đầu tiến qua sông . Quân cùa Nguyễn__ Hữu _ Tiến phải lùi . Quân Nguyễn__ phải lui về đóng ở Nghi _ Xuân . Chúa _ Hiền mang quân tiếp _ ứng đóng ở làng Phù _ Lộ ( làng Bình _ An , huyện Bình _ Chính , tinh Quảng _ Bình ) . Chúa _ Trịnh _ Căn dẫn quân về Thăng _ Long , sai Đào _ Quang _ Nhiêu ở lại trấn _ thủ Nghệ _ An và chiếm _ giữ châu Bắc _ Bố _ Chính ’ . Hữu _ Dật đã cho sửa _ sang thành _ luỹ , vỗ yên quân _ dân , phòng giữ biên _ cương càng thêm vững _ chắc đề _ phòng quân Trịnh . Hữu _ Dật sai Trương _ Vàn _ Vân và Vân _ Trạch chia quân ra chống _ cự . Tướng Trịnh là Hoan _ Trung đem quân và tán lọng của vua ra chống _ cự nhưng không được , Hoan _ Trung bị chết , quân Trịnh sợ bỏ chạy . Chúa _ Trịnh _ Căn giữ chức Nguyên _ soái thuỷ _ quân , Lê _ Thì Hiến giữ chức_Thống _ suất quân bộ . Được tin báo của Trấn _ thủ châu Bố _ Chính là Triều _ Tín , chúa Hiền liền cử con thứ tư là Chường cơ Hiệp _ đức hầu làm Nguyên _ soái , Nha uý Mai _ Phú _ Lĩnh và Ký _ lục Vũ _ Phi _ Thừa làm Tham _ mưu , Chưởng cơ Trương _ Phúc _ Cương ( con Trương _ Phúc _ Phấn ) và Nguyễn__ Đức _ Bảo làm Tả _ hữu tiên _ phong đi đánh quân Trịnh . Tháng _ Bày , Nguyên _ soái Hiệp bat đầu xuất _ quân . Nguyễn__ Hữu _ Dật giữ luỹ Sa _ Phụ , Trấn thù Quảng _ Bình là Nguyễn__ Mỹ _ Đức giữ Chính Luỹ , Chưởng cơ Trương _ Phúc _ Cương giữ luỹ Irán _ Ninh , Iran thú Bố Chinh lã Irièu lin giữ luỹ Động Hồi , Iran thủ Cựu _ Dinh là Thuận _ Đức giữ luỹ Đâu _ Mâu , Cai cơ Thuận _ Trung giữ cầu Mỗi Nại , Tham tướng Tài _ Lễ đem chiến _ thuyền đóng cọc gỗ ngàn cửa _ biển Nhật _ Lệ . Chúa _ Hiền thân đốc _ suất đại _ quân thuỷ _ bộ cùng tiến . Quân Nguyễn__ cho lính đánh _ úp , quân Trịnh phải lui . Quân Nguyễn__ ờ trên luỹ bày súng bắn xuống . Quân Trịnh bám vào đông như kiến leo lên . Quân Nguyễn__ chụm mác vào đâm . Hữu _ Dật đến luỹ đã bị phá vỡ 30 trượng , hầu _ như không _ thể chống đờ được nữa . Quân Trịnh và quân Nguyễn__ đánh nhau kịch _ liệt mấy ngày trời , xác quân Trịnh chất thành đống , quân Nguyễn_cũng bị _ thương và chết rất nhiều . Thì Hiến liền ngay _ lập _ tức đánh luỹ . Nguyễn__ Hữu _ Dật cố sức giữ luỹ , " nhuệ _ khí gấp mười " ’ . Thì Hiến không _ thể đánh được . Trịnh _ Tạc lại dẫn vua Lê_về Đông _ Đô . Tướng Trịnh đang trấn _ thủ Nghệ _ An là Đào _ Quang _ Nhiêu thì bị chết . Lấy sông Gianh làm giới _ tuyến ’ . ( XX ) đóng ờ biên _ giới phía Bắc , 9.000 ở triều _ đình , 6.0 ( X ) lệ _ thuộc vào các hoàng _ tử , 15 . ( X ) 0 quân đóng ở các địa _ phương ) . Hơn _ nữa , quân Nguyễn__ không phải đi xa đã làm chủ được chiến _ trường của mình . Như _ vậy , về binh _ lực tuy họ Trịnh có hơn họ Nguyễn__ nhưng cuộc nội _ chiến đã kết _ thúc với sự tấn _ công thất _ bại của quân Trịnh và sự chống ưả thắng _ lợi của quân Nguyễn__ . Nguyên _ nhân thất _ bại của quân Trịnh trước _ hết là do phải hành _ quân chiến _ đấu xa căn _ cứ , đường _ sá chuyển lương và hành _ quân hết _ sức khó _ khăn . Lương cung _ cấp cho quân _ lính thường không chuyển kịp ."""

summary = extractive_summary(
    text,
    window_size=10,
    stride=7,
    extract_ratio=0.5
)

print("📌 TÓM TẮT TRÍCH XUẤT:")
print(summary)
    

📌 TÓM TẮT TRÍCH XUẤT:
Sứ họ Trịnh đành phải quay về . Quân hai bên đối luỹ nhau cùng cầm _ cự . Cánh quân _ chủ _ lực của Trịnh _ Tráng đang rất mạnh vừa kéo đến , quân Nguyễn__ _ đánh không lại . Quân Trịnh thừa _ thắng cướp của _ cải . Sau đó quân Nguyễn__ đem tượng _ binh đánh chặn ngang làm quân Trịnh tan võ , quân _ lính chết rất nhiều . Nguyễn__ _ Hữu _ Dật mưu vói Truong _ Phước _ Da sai gián _ điệp phao _ đồn rằng anh _ em Trịnh _ Gia , Trịnh _ Nhạc mưu nổi _ loạn . Nghe vậy , chúa Sãi quyết _ định tạm nhận sắc để họ Trịnh không ngờ và có thời _ gian phòng _ thủ chuẩn _ bị lực _ lượng đối _ phó rồi sẽ dùng kế trả lại sắc . Khi lực _ lượng đã được chuẩn _ bị , phòng _ tuyến tự _ vệ đã hoàn _ thiện , Nguyễn__ _ Phúc _ Nguyên đã cùng với Đào _ Duy _ Từ ngày _ đêm mưu _ tính chống họ Trịnh ' . Chúa _ Nguyễn__ đã cho mộ thêm những người khoẻ _ mạnh làm _ thân _ binh . Anh tức _ thì sai người đi quy _ thuận với họ Trịnh * . Nguyễn__ _ Phúc _ Nguyên sai Đại _ tướng Nguyễn__ _ Mỹ _ Thắ

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import pandas as pd
from vncorenlp import VnCoreNLP
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

vncorenlp = VnCoreNLP(
    "/kaggle/input/vncorenlp/pytorch/default/1/VnCoreNLP-1.2.jar",
    annotators="wseg"
)

annotator = VnCoreNLP(
    "/kaggle/input/vncorenlp/pytorch/default/1/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

class SimpleLLM:
    def __init__(self, model, tokenizer, max_new_tokens=512):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens

    def __call__(self, prompt: str) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
llm = SimpleLLM(model, tokenizer)

def summary_abstractive(text, grade, length):
    prompt = [
        SystemMessage(content=f"Bạn là giáo viên dạy môn học tiếng Việt cho học sinh tiểu học lớp {grade} tại Việt Nam."),
        HumanMessage(content=f"""
        Đầu tiên tôi muốn bạn biết tóm tắt diễn giải là gì ?. Tóm tắt diễn giải (Abstractive Summarization): là phương pháp tóm tắt trong đó hệ thống hiểu toàn bộ nội dung tổng thể và tái diễn đạt lại các ý chính ngắn gọn và tự nhiên trong một văn bản mới, tương tự như cách con người tóm tắt. Hệ thống sẽ tạo ra các câu văn ngắn gọn, mạch lạc, tự nhiên và dễ hiểu.
        Tiếp theo tôi muốn bạn đọc đoạn văn sau và thực hiện tóm tắt diễn giải văn bản sao cho phù hợp với học sinh lớp {grade}.
        Với các yêu cầu như sau:
        - Tóm tắt diễn giải phải đảm bảo tính chính xác, tính logic, tính hợp lý, không mất ý nghĩa của văn bản gốc.
        - HÃY TUÂN THỦ VÀ THỰC HIỆN TÓM TẮT VĂN BẢN TRONG {length} CÂU. Phải chú trọng đến độ liên kết giữa các câu, giữa các đoạn văn. Nội dung mạch lạc, trôi chảy, liên kết. Bản tóm tắt phải dễ hiểu, lời lẽ tự nhiên, từ vựng phù hợp, đơn giản với học sinh lớp {grade}.
        - Cuối cùng hãy chuẩn hóa bản tóm tắt thành dạng văn bản hoàn chỉnh. Viết trên cùng 1 dòng.
        - KHÔNG THÊM CÁC DÁNH DẤU, CHỈ MỤC, CHÚ THÍCH, GHI CHÚ, ...
        - KẾT QUẢ CHỈ GỒM VĂN BẢN ĐƯỢC TÓM TẮT. KHÔNG THÊM TẤT CẢ CÁC YẾU TỐ, THÔNG TIN NGOÀI LỀ KHÁC.
        ĐOẠN VĂN:
        {text}

        KẾT QUẢ:
        """)
    ]
    response = llm(prompt)
    return response.content.strip()

def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)
    tokens = []
    for sent in sentences:
        tokens.extend(sent)
    return tokens

def load_grade_vocab(grade, vocab_dir="/kaggle/working/data"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab

def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0
    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)

text = """Bức thư này là lời động viên của Chủ tịch Hồ Chí Minh gửi các em học sinh nhân ngày khai trường đầu tiên của nước Việt Nam Dân chủ Cộng hòa. Bác Hồ đã bày tỏ niềm vui mừng khi thấy các em được trở lại trường học sau những tháng ngày nghỉ ngơi và biến động. Bác cũng nhấn mạnh rằng, từ nay trở đi, các em sẽ được hưởng một nền giáo dục hoàn toàn Việt Nam, một điều mà trước đây không thể có được.
Bác Hồ mong muốn các em học sinh hãy cố gắng học tập, rèn luyện đạo đức, trở thành những người công dân có ích cho đất nước. Bác tin rằng, tương lai của đất nước phụ thuộc rất lớn vào sự nỗ lực học tập của các em. Bác cũng gửi lời chúc tốt đẹp nhất đến các em nhân dịp năm học mới."""
content = text
grade = 5
sentences = annotator.tokenize(content)
num_sentences = len(sentences)
length = num_sentences // 3
print(summary_abstractive(content, grade, length))

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import pandas as pd
import os
from vncorenlp import VnCoreNLP
from kaggle_secrets import UserSecretsClient
import math
import re

user_secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-4.1-mini",   # rẻ + đủ mạnh tiếng Việt
    temperature=0.7,
    top_p=0.9,
    max_tokens=512
)

vncorenlp = VnCoreNLP(
    "/kaggle/input/vncorenlp/pytorch/default/1/VnCoreNLP-1.2.jar",
    annotators="wseg"
)

annotator = VnCoreNLP(
    "/kaggle/input/vncorenlp/pytorch/default/1/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

SYSTEM_PROMPT = "Bạn là giáo viên tiếng Việt tiểu học tại Việt Nam."

def summary_abstractive(text, grade, length):
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(
            content=(
                f"Tóm tắt đoạn văn sau cho học sinh lớp {grade}. "
                f"Viết đúng {length} câu, ngôn ngữ đơn giản, tự nhiên, phù hợp học sinh. Đảm bảo đủ ý, chính xác, mạch lạc. Viết trên một dòng, không thêm tiêu đề, chú thích, ký hiệu, lời dẫn.\n\n"
                f"{text}"
            )
        )
    ]
    response = llm.invoke(messages)
    return response.content.strip()

def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)
    tokens = []
    for sent in sentences:
        tokens.extend(sent)
    return tokens

def load_grade_vocab(grade, vocab_dir="/kaggle/working/data"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab

def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0
    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)

VOCAB_CACHE = {}

def get_vocab_cached(grade):
    if grade not in VOCAB_CACHE:
        VOCAB_CACHE[grade] = load_grade_vocab(grade)
    return VOCAB_CACHE[grade]

SAVE_EVERY = 5000
OUTPUT_PATH = "/kaggle/working/data/datadgnew/DATA_DG_output_safe1.xlsx"
data = pd.read_excel("/kaggle/working/data/datadgnew/DATA_DG.xlsx")
df = data[:8000]
max_retry = 1
threshold = 0.9
processed = 0

for idx, row in df.iterrows():

    existing_summary = row.get("summary", "")
    if pd.notna(existing_summary) and str(existing_summary).strip():
        continue

    content = str(row["content"]).strip()
    if not content:
        df.loc[idx, "summary"] = ""
        continue

    grade = int(row["grade"])

    try:
        sentences = annotator.tokenize(content)
        num_sentences = len(sentences)
    except Exception:
        df.loc[idx, "summary"] = ""
        continue

    length = max(1, math.ceil(num_sentences * 0.5))
    vocab = get_vocab_cached(grade)

    summary = summary_abstractive(content, grade, length)
    score = vocab_coverage(summary, vocab)

    df.loc[idx, "summary"] = summary if score >= threshold else ""
    processed += 1

    if processed % SAVE_EVERY == 0:
        df.to_excel(OUTPUT_PATH, index=False)
        print(f"✅ Đã lưu tạm sau {processed} dòng")

df.to_excel(OUTPUT_PATH, index=False)
print("🎉 Hoàn tất")



In [4]:
import pandas as pd

# Đọc file
df1 = pd.read_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_main_3.xlsx"
)
df2 = pd.read_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_output_safe1 (1).xlsx"
)

# Chuẩn hoá content (TRIM + bỏ space thừa)
def normalize(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

df1["content_norm"] = df1["content"].apply(normalize)
df2["content_norm"] = df2["content"].apply(normalize)

# Tập content của df1
content_set = set(df1["content_norm"])

# Giữ lại df2 nếu content có trong df1
df2_filtered = df2[df2["content_norm"].isin(content_set)].copy()

# Xoá cột phụ
df2_filtered.drop(columns="content_norm", inplace=True)

# Lưu file
df2_filtered.to_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_output_safe1_filtered.xlsx",
    index=False
)


In [5]:
df1 = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_output_safe1_filtered.xlsx")
df1 = df1[df1["summary"].notna()].copy()
df1.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_output_safe1_final.xlsx")

In [1]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from underthesea import word_tokenize
import torch

# ---------------------------
# Tiền xử lý tiếng Việt
# ---------------------------
def preprocess_vi(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate, reference):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=False
    )

    scores = scorer.score(reference, candidate)

    rouge_results = {
        "rouge1_precision": scores['rouge1'].precision,
        "rouge1_recall": scores['rouge1'].recall,
        "rouge1_f1": scores['rouge1'].fmeasure,
        "rougeL_precision": scores['rougeL'].precision,
        "rougeL_recall": scores['rougeL'].recall,
        "rougeL_f1": scores['rougeL'].fmeasure,
    }

    return rouge_results


# ---------------------------
# Tính BERTScore (PhoBERT)
# ---------------------------
from bert_score import score
import torch

def compute_bertscore(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = score(
        [candidate],
        [reference],
        lang="vi",          # QUAN TRỌNG
        model_type=None,    # KHÔNG ghi vinai/phobert-base
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }



# ---------------------------
# Hàm tổng hợp
# ---------------------------
def evaluate_summary(candidate, reference):
    candidate_proc = preprocess_vi(candidate)
    reference_proc = preprocess_vi(reference)

    candidate_text = " ".join(candidate_proc)
    reference_text = " ".join(reference_proc)

    results = {}
    results.update(compute_rouge(candidate_text, reference_text))
    results.update(compute_bertscore(candidate_text, reference_text))

    return results




In [3]:
import pandas as pd
import numpy as np

data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_work.xlsx")
df = data[:4000].copy()
# Danh sách cột thang đo
metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1"
]

# Nếu chưa có cột thì tạo
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

# =========================
# TÍNH ĐIỂM TỪNG ROW
# =========================
for idx, row in df.iterrows():
    content = row.get("content", "")
    summary = row.get("summary", "")

    # Bỏ qua nếu thiếu dữ liệu
    if not isinstance(content, str) or not isinstance(summary, str):
        continue
    if content.strip() == "" or summary.strip() == "":
        continue

    try:
        scores = evaluate_summary(summary, content)

        for k, v in scores.items():
            df.at[idx, k] = float(v)

    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue
    
df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_work_score.xlsx", index=False)


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
df.describe()

,grade,rouge1_precision,rouge1_recall,rouge1_f1,rougeL_precision,rougeL_recall,rougeL_f1,bertscore_precision,bertscore_recall,bertscore_f1
count,4000.000000,3486.000000,3486.000000,3486.000000,3486.000000,3486.000000,3486.000000,3486.000000,3486.000000,3486.000000
mean,3.314500,0.977812,0.551479,0.682652,0.787697,0.435665,0.542045,0.837435,0.823718,0.830462
std,1.069054,0.045429,0.183622,0.158867,0.101221,0.139474,0.122641,0.045419,0.051972,0.048428
min,1.000000,0.380074,0.071924,0.134197,0.309963,0.070032,0.130659,0.694254,0.666761,0.680968
25%,3.000000,0.975816,0.439151,0.609186,0.731412,0.353657,0.482477,0.792416,0.774761,0.782739
50%,3.000000,0.995175,0.553557,0.708504,0.800000,0.433120,0.546601,0.858196,0.844609,0.852026
75%,4.000000,1.000000,0.675297,0.794772,0.856388,0.517518,0.614373,0.873891,0.867357,0.870145
max,5.000000,1.000000,1.000000,0.979167,1.000000,1.000000,0.979167,1.000000,1.000000,1.000000


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import pandas as pd
import os
from vncorenlp import VnCoreNLP
from dotenv import load_dotenv
import os
load_dotenv()
llm = ChatOpenAI(
    model="gpt-4o-mini",   # rẻ + đủ mạnh tiếng Việt
    temperature=0.7,
    top_p=0.9,
    max_tokens=512
)

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import pandas as pd
from vncorenlp import VnCoreNLP
from dotenv import load_dotenv
import os
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",   # rẻ + đủ mạnh tiếng Việt
    temperature=0.7,
    top_p=0.9,
    max_tokens=512
)

vncorenlp = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg"
)

annotator = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

def summary_abstractive(text, grade, length):
    messages = [
        SystemMessage(
            content=f"Bạn là giáo viên dạy tiếng Việt cho học sinh tiểu học lớp {grade} tại Việt Nam."
        ),
        HumanMessage(
            content=f"""
NHIỆM VỤ:
Tóm tắt diễn giải đoạn văn sau cho học sinh lớp {grade}.

YÊU CẦU BẮT BUỘC:
- Viết ĐÚNG {length} câu
- Giữ đầy đủ các ý chính, ý phụ của văn bản gốc
- Ngôn ngữ đơn giản, tự nhiên, phù hợp
- Các câu đảm bảo chính xác, logic, mạch lạc
- Viết trên MỘT DÒNG
- KHÔNG thêm tiêu đề, chú thích, ký hiệu, lời dẫn hay nhận xét.

ĐOẠN VĂN:
{text}

KẾT QUẢ:
"""
        )
    ]

    response = llm.invoke(messages)
    return response.content.strip()

def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)
    tokens = []
    for sent in sentences:
        tokens.extend(sent)
    return tokens

def load_grade_vocab(grade, vocab_dir="../../../datasets/grade_vocab"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab

def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0
    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)

In [8]:
test_text = """Hanami là lễ hội Hoa anh đào truyền thống của Nhật Bản. Lễ hội này diễn ra từ tháng 3 đến tháng 5 hằng năm khi hoa anh đào nở rộ. Vào những ngày lễ hội, du khách đến đây có thể tham gia nhiều hoạt động thú vị: 
Đi dạo hoặc bơi thuyền ngắm hoa anh đào. 
Tổ chức tiệc trà trong vườn hoa anh đào. 
Thưởng thức ẩm thực Nhật Bản, trong đó có nhiều món được chế biến từ hoa anh đào. 
Ca hát hoặc giao lưu văn hoá nghệ thuật truyền thống mừng mùa hoa anh đào nở."""
print(summary_abstractive(test_text, grade=4, length=3))

Hanami là lễ hội Hoa anh đào truyền thống của Nhật Bản, diễn ra từ tháng 3 đến tháng 5 khi hoa nở rộ. Trong lễ hội, du khách có thể tham gia nhiều hoạt động như đi dạo, bơi thuyền ngắm hoa, tổ chức tiệc trà, thưởng thức ẩm thực Nhật Bản và giao lưu văn hoá nghệ thuật. Đây là dịp để mọi người cùng vui chơi và thưởng thức vẻ đẹp của hoa anh đào.


In [4]:
import math
import re

SAVE_EVERY = 200
OUTPUT_PATH = "E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_work_summary.xlsx"
df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_work_new.xlsx")

max_retry = 1
threshold = 0.83
processed = 0

for idx, row in df.iterrows():

    # 👉 CHỈ xử lý nếu summary rỗng
    existing_summary = row.get("summary", "")
    if pd.notna(existing_summary) and str(existing_summary).strip():
        continue

    content = str(row["content"]).strip()
    if not content:
        df.loc[idx, "summary"] = ""
        continue

    grade = int(row["grade"])

    # Đếm số câu
    try:
        sentences = annotator.tokenize(content)
        num_sentences = len(sentences)
    except Exception:
        df.loc[idx, "summary"] = ""
        continue

    length = max(1, math.ceil(num_sentences * 0.5))

    vocab = load_grade_vocab(grade)

    best_summary = ""
    best_score = 0.0

    for _ in range(max_retry):
        summary = summary_abstractive(content, grade, length)
        score = vocab_coverage(summary, vocab)

        if score > best_score:
            best_score = score
            best_summary = summary

        if best_score >= threshold:
            break

    df.loc[idx, "summary"] = best_summary if best_score >= threshold else ""

    processed += 1

    if processed % SAVE_EVERY == 0:
        df.to_excel(OUTPUT_PATH, index=False)
        print(f"✅ Đã lưu tạm sau {processed} dòng")

# 👉 LƯU LẦN CUỐI
df.to_excel(OUTPUT_PATH, index=False)
print("🎉 Hoàn tất, đã lưu file cuối cùng")


✅ Đã lưu tạm sau 200 dòng
✅ Đã lưu tạm sau 400 dòng
✅ Đã lưu tạm sau 600 dòng
✅ Đã lưu tạm sau 800 dòng
✅ Đã lưu tạm sau 1000 dòng
✅ Đã lưu tạm sau 1200 dòng
✅ Đã lưu tạm sau 1400 dòng
✅ Đã lưu tạm sau 1600 dòng
✅ Đã lưu tạm sau 1800 dòng
✅ Đã lưu tạm sau 2000 dòng
✅ Đã lưu tạm sau 2200 dòng
🎉 Hoàn tất, đã lưu file cuối cùng


In [2]:
import pandas as pd 
df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\DATA_DG_work_new.xlsx")
df

,title,content,summary,grade,publisher
0,Thi vẽ,Cá chép và gà nhí thi vẽ.\nCá chép vẽ nó làm v...,"Cá chép và gà nhí thi vẽ, cá chép vẽ làm vua c...",1,CÁNH DIỀU
1,"Lúa nếp, lúa tẻ",Lúa tẻ cho là nó thua kém lúa nếp vì trẻ em ch...,Lúa tẻ nghĩ rằng lúa nếp hơn mình vì trẻ em th...,1,CÁNH DIỀU
2,Sẻ và cò,Sẻ gặp cò ở hồ. Sẻ chê mỏ cò thô. Cò chả đáp g...,Sẻ gặp cò ở hồ và chê mỏ cò thô nhưng cò không...,1,CÁNH DIỀU
3,Đêm ở quê,"Đêm ở quê quả là êm ả.\nỞ thị xã, cả đêm ì ầm ...","Đêm ở quê rất yên tĩnh chỉ nghe tiếng gió, tiế...",1,CÁNH DIỀU
4,Gà nhí nằm mơ,"Trưa hè, gà nhí nằm mơ bị quạ cắp đi.\nGà nhí ...","Trưa hè, gà nhí mơ thấy quạ cắp đi nên sợ và k...",1,CÁNH DIỀU
...,...,...,...,...,...
22440,Tứ đại mỹ nhân thời cổ Trung Quốc là những ai?,Lịch sử Trung Quốc có bốn người đẹp làm khuynh...,Lịch sử Trung Quốc có bốn người đẹp nổi tiếng ...,5,NaN
22441,Đạo giáo đã nảy sinh như thế nào?,"Đạo giáo (hay Lão giáo), Phật giáo và Hồi giáo...","Đạo giáo, Phật giáo và Hồi giáo là ba tôn giáo...",5,NaN
22442,Tại sao bệnh dịch hạch lại trở thành đại hoạ c...,"Ngày 24 tháng 3 năm 1345, người ta phát hiện t...","Vào ngày 24 tháng 3 năm 1345, một hiện tượng k...",5,NaN
22443,Đế quốc Ôt-tôman ra đời như thế nào?,"Giữa thế kỷ XIII, Tiểu Á rơi vào tình trạng hỗ...",NaN,5,NaN
